# Cattle disease classifier

This notebook trains a three-class classifier for **foot-and-mouth**, **healthy**, and **lumpy** images. Run every code cell from top to bottom. It saves `cattle_disease_model.keras` and `class_names.json`, which are used by `app.py`.

Important: a model can only be as reliable as its labelled images. Check labels and use the held-out test results before relying on it.

In [3]:
from pathlib import Path
import json
import random
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

# If Jupyter is opened elsewhere, replace Path.cwd() with this folder's path.
PROJECT_DIR = Path.cwd()
CLASS_NAMES = ['foot-and-mouth', 'healthy', 'lumpy']
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

files_by_class = {}
for class_name in CLASS_NAMES:
    folder = PROJECT_DIR / class_name
    if not folder.is_dir():
        raise FileNotFoundError(f'Missing class folder: {folder}')
    files = sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in EXTENSIONS)
    if len(files) < 10:
        raise ValueError(f'{class_name} needs at least 10 images; found {len(files)}')
    files_by_class[class_name] = files
    print(f'{class_name}: {len(files)} images')

# Stratified 70% train, 15% validation, 15% test split.
train_paths, train_labels, val_paths, val_labels, test_paths, test_labels = [], [], [], [], [], []
rng = random.Random(SEED)
for label, class_name in enumerate(CLASS_NAMES):
    paths = files_by_class[class_name].copy()
    rng.shuffle(paths)
    n_train, n_val = int(.70 * len(paths)), int(.15 * len(paths))
    splits = ((paths[:n_train], train_paths, train_labels),
              (paths[n_train:n_train+n_val], val_paths, val_labels),
              (paths[n_train+n_val:], test_paths, test_labels))
    for selected, output_paths, output_labels in splits:
        output_paths.extend(map(str, selected))
        output_labels.extend([label] * len(selected))

print(f'Train: {len(train_paths)}, validation: {len(val_paths)}, test: {len(test_paths)}')

foot-and-mouth: 746 images
healthy: 1291 images
lumpy: 1206 images
Train: 2269, validation: 484, test: 490


In [4]:
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label):
    image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.cast(image, tf.float32)
    # This must also be used in app.py. MobileNetV2 requires [-1, 1].
    image = tf.keras.applications.mobilenet_v2.preprocess_input(image)
    return image, label

def make_dataset(paths, labels, training=False):
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        dataset = dataset.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    return (dataset.map(load_image, num_parallel_calls=AUTOTUNE)
                   .batch(BATCH_SIZE)
                   .prefetch(AUTOTUNE))

train_ds = make_dataset(train_paths, train_labels, training=True)
val_ds = make_dataset(val_paths, val_labels)
test_ds = make_dataset(test_paths, test_labels)

counts = np.bincount(train_labels, minlength=len(CLASS_NAMES))
class_weight = {i: len(train_labels) / (len(CLASS_NAMES) * count) for i, count in enumerate(counts)}
print('Class weights:', class_weight)

augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.10),
])
base = tf.keras.applications.MobileNetV2(include_top=False, weights='imagenet', input_shape=(224, 224, 3))
base.trainable = False
inputs = tf.keras.Input(shape=(224, 224, 3))
x = augmentation(inputs)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.35)(x)
outputs = tf.keras.layers.Dense(len(CLASS_NAMES), activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

checkpoint = PROJECT_DIR / 'best_cattle_disease_model.keras'
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', patience=2, factor=0.3, min_lr=1e-7),
    tf.keras.callbacks.ModelCheckpoint(checkpoint, monitor='val_loss', save_best_only=True),
]
history = model.fit(train_ds, validation_data=val_ds, epochs=15, class_weight=class_weight, callbacks=callbacks)

Class weights: {0: 1.4489144316730524, 1: 0.837578442229605, 2: 0.8961295418641391}
Epoch 1/15
71/71 [==============================] - 100s 1s/step - loss: 1.4000 - accuracy: 0.3645 - val_loss: 1.0575 - val_accuracy: 0.4814 - lr: 1.0000e-04
Epoch 2/15
71/71 [==============================] - 96s 1s/step - loss: 1.1074 - accuracy: 0.4799 - val_loss: 0.8104 - val_accuracy: 0.6529 - lr: 1.0000e-04
Epoch 3/15
71/71 [==============================] - 83s 1s/step - loss: 0.9152 - accuracy: 0.5901 - val_loss: 0.6824 - val_accuracy: 0.7293 - lr: 1.0000e-04
Epoch 4/15
71/71 [==============================] - 82s 1s/step - loss: 0.7932 - accuracy: 0.6593 - val_loss: 0.5829 - val_accuracy: 0.7686 - lr: 1.0000e-04
Epoch 5/15
71/71 [==============================] - 88s 1s/step - loss: 0.7080 - accuracy: 0.6844 - val_loss: 0.5304 - val_accuracy: 0.7851 - lr: 1.0000e-04
Epoch 6/15
71/71 [==============================] - 86s 1s/step - loss: 0.6588 - accuracy: 0.7100 - val_loss: 0.4907 - val_accurac

In [ ]:
# Fine-tune only the last MobileNetV2 layers at a small learning rate.
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_finetune = model.fit(train_ds, validation_data=val_ds, epochs=10, class_weight=class_weight, callbacks=callbacks)

# Restore the best validation model, then evaluate once on untouched test images.
model = tf.keras.models.load_model(checkpoint)
test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)
y_true = np.concatenate([labels.numpy() for _, labels in test_ds])
y_pred = np.argmax(model.predict(test_ds, verbose=0), axis=1)
print(f'Test accuracy: {test_accuracy:.2%}; test loss: {test_loss:.4f}')
print('Confusion matrix:\n', confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

# Save the model and its exact label order together.
model.save(PROJECT_DIR / 'cattle_disease_model.keras')
(PROJECT_DIR / 'class_names.json').write_text(json.dumps(CLASS_NAMES, indent=2), encoding='utf-8')
print('Saved:', PROJECT_DIR / 'cattle_disease_model.keras')
print('Saved:', PROJECT_DIR / 'class_names.json')

Epoch 1/10
71/71 [==============================] - 115s 1s/step - loss: 0.4272 - accuracy: 0.8233 - val_loss: 0.2609 - val_accuracy: 0.9050 - lr: 1.0000e-05
Epoch 2/10
71/71 [==============================] - 102s 1s/step - loss: 0.3457 - accuracy: 0.8656 - val_loss: 0.2358 - val_accuracy: 0.9091 - lr: 1.0000e-05
Epoch 3/10
71/71 [==============================] - 101s 1s/step - loss: 0.2957 - accuracy: 0.8792 - val_loss: 0.2053 - val_accuracy: 0.9298 - lr: 1.0000e-05
Epoch 4/10
71/71 [==============================] - 94s 1s/step - loss: 0.2724 - accuracy: 0.8863 - val_loss: 0.1934 - val_accuracy: 0.9339 - lr: 1.0000e-05
Epoch 5/10
71/71 [==============================] - 103s 1s/step - loss: 0.2394 - accuracy: 0.9039 - val_loss: 0.1976 - val_accuracy: 0.9318 - lr: 1.0000e-05
Epoch 6/10
71/71 [==============================] - 96s 1s/step - loss: 0.2356 - accuracy: 0.9044 - val_loss: 0.2009 - val_accuracy: 0.9318 - lr: 1.0000e-05
Epoch 7/10
71/71 [==============================] - 96

: 